# பாடம் 10 - உற்பத்தியில் AI முகவர்கள்

இந்த பாடத்தில் Microsoft Agent Framework (Python) பயன்படுத்தி AI முகவர்களுக்கு உள்ள **உற்பத்தி மாதிரிகள்** பற்றி நீங்கள் கற்றுக்கொள்ளப்போகிறீர்கள். இதில் நாம் விரிவாகப் பேசுகிறோம்:

- **கண்டறிதல்** — முகவர் தொடர்புகளில் நேரம் மற்றும் பதிவு சேர்க்கிறது
- **மதிப்பீடு** — பதில் தரத்தைக் கணிக்க ஒரு மதிப்பாய்வாளர் முகவரைப் பயன்படுத்துதல்
- **செலவு மேலாண்மை** — டோக்கன் முறைப்பாடு மற்றும் மாதிரி தேர்வுக்கான உத்திகள்

இந்த நடைமுறை ஒரு **பயண முகவர்** குறித்ததாகும், இது பயனர்களுக்கு பயணங்களைத் திட்டமிட உதவுகிறது, மேலும் கண்காணிப்பு மற்றும் மதிப்பீடு மேலேயுள்ளது.


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## உற்பத்தி கருத்துக்கள்

AI முகவர்களை மாதிரிகளிலிருந்து உற்பத்திக்கு மாற்றுவது மூன்று முக்கியக் தளங்கள் மீது கவனம் செலுத்த வேண்டும்:

1. **கண்காணிப்பு** — முகவர் என்ன செய்கிறான், எவ்வளவு காலம் எடுப்பான், எந்த கருவிகளை அழைக்கிறான் என்பதில் தெளிவு வேண்டும். டிரேஸிங் மற்றும் பதிவு இல்லாமல், உற்பத்தி பிரச்சனைகளை பற்றி கண்டுபிடிப்பது சுமார் அசாத்தியமாகிறது.

2. **மதிப்பீடு** — தானியங்கு தர நீட்டிப்புகள் முகவரின் பதில்கள் நேர்த்தியான, முழுமையான, மற்றும் உதவிக்கரமானவை என்றும் நிலைநிறுத்துகின்றன. ஒரு மதிப்பாய்வாளர் முகவர் வரையறுக்கப்பட்ட நிபந்தனைகளுக்கு எதிராக பதில்களை மதிப்பீடு செய்யலாம்.

3. **செலவு மேலாண்மை** — டோக்கன் பயன்பாடு நேரடியாக செலவுக்கு பாதிப்பை ஏற்படுத்தும். கேள்வி சிறப்புறுத்தல், மாதிரி தேர்வு, மற்றும் காசிங் போன்ற கொள்கைகள் செலவுகளை கட்டுப்படுத்த உதவுகின்றன, தரத்தை தள்ளுபடி செய்யாமல்.


## ஒரு பார்வையிடக்கூடிய முகவரியை உருவாக்குதல்

நாம் பயண கருவிகளை வரையறுக்கின்றோம் மற்றும் முகவரியின் அழைப்பை நேரத்துடன் சுற்றி வைத்து, தாமதத்தை கண்காணிக்க முடியும். உற்பத்தியில் நீங்கள் OpenTelemetry அல்லது இதே போன்ற தடயம் பின்கண்காணிப்பு பின்னணியுடன் ஒருங்கிணைக்கும்.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## மதிப்பீட்டு மாதிரிகள்

ஒரு பொதுவான தயாரிப்பு முறையானது இரண்டாவது முகவரியை **மதிப்பாய்வாளர்** ஆக பயன்படுத்துவது. மதிப்பாய்வாளர் முதன்மையான முகவரியின் பதிலை முழுமை, துல்லியம் மற்றும் உதவித்தன்மை போன்ற முன்கூட்டிய criterios எதிராக மதிப்பிடுகிறார்.

இது இயலும்:
- பதில்கள் பயனர்களுக்கு செல்லும் முன் தானியங்கி தரநிலை வாயில்கள்
- செய்யுள் அல்லது மாதிரிகள் மாறும்போது மீளாய்வு கண்டறிதல்
- காலப்போக்கில் முகவரின் செயல்திறன் தொடர்ச்சியான கண்காணிப்பு


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## செலவு மேலாண்மை.Strategies

உற்பத்தி AI முகவரிகளுக்கான செலவுகளை கட்டுப்படுத்துவது மிகவும் அவசியம். இங்கே முக்கியமான நகரச்சிக்கல்கள் கொடுக்கப்பட்டுள்ளன:

| நகரச்சிக்கை | விளக்கம் |
|---|---|
| **அறுவை விளக்கத்தின் மேம்பாடு** | கணினிச்செய்தி கட்டளைகள் சுருக்கமாக இருக்க வேண்டும். உள்ளீட்டு டோக்கன்களை குறைக்க தேவையில்லாத உள்ளடக்கத்தை அகற்று. |
| **மாதிரி தேர்வு** | எளிமையான பணிகளுக்கு (உதாரணம்: வகை அடுக்கல் அல்லது எடுத்துக்காட்டுதல்) சிறிய, குறைந்த செலவுடைய மாதிரிகளை (எ.கா GPT-4o-mini) பயன்படுத்தவும், மற்றும் சிக்கலான கருதி நடவடிக்கைகளுக்கு பெரிய மாதிரிகளை பதிவு செய்யவும். |
| **தரவு சேமிப்பு** | கருவி முடிவுகள் மற்றும் அடிக்கடி வரும் கேள்விகளை சேமித்து, மீண்டும் API அழைப்புகளைத் தவிர்க்கவும். |
| **டோகன் பட்ஜெட்டுகள்** | எதிர்பாராத நீண்ட பதில்களைத் தடுப்பதற்கு `max_tokens` நிலைகள் அமைக்கவும். |
| **தொகுப்பு** | பல பயனர் கேள்விகளை சாத்தியமான இடத்தில் ஒரு API அழைப்பாகக் குழுவாக்கவும். |

விளக்கத்தில், ஒரு படிநிலை அணுகுமுறை நன்றாக செயல்படுகிறது: நேரடியான கேள்விகளை விரைவான, காசு குறைந்த மாதிரிக்கு வழிமாற்று செய்து, சிக்கலான கேள்விகளை மட்டும் திறமையான மாதிரிக்குக் கொண்டு செல்லவும்.


## Summary

இலக்கினில் நீங்கள் כיצדஇவற்றை கற்றுக்கொண்டீர்கள்:

1.  **கணக்கிடக்கூடிய தன்மையைச் சேர்க்கவும்** நேர ஒத்திசைவு மற்றும் பதிவு செய்வதன் மூலம் முகவர் தொடர்புகளுக்கு, தடையியல் மற்றும் கண்காணிப்புக்கான அடிப்படையை அமைத்தல்.
2.  **முகவர் பதில்களை மதிப்பீடு செய்யவும்** ஒரு மதிப்பாய்வு முகவரைக் கொண்டு, முழுமை, துல்லியம் மற்றும் உதவிப்பொருள் ஆகியவற்றுக்கான மதிப்பெண்களை சுயமாக கணக்கிடுதல்.
3.  **செலவுகளை நிர்வகிக்கவும்** விரிவாக்கம், மாதிரி தேர்வு, தளர்வு மற்றும் குறியீட்டு பட்ஜெட்டுகள் மூலம்.

இந்த உற்பத்தி முறைகள் உங்கள் AI முகவர்களை நம்பகமானதாக, அளவிடக்கூடியதாக மற்றும் பெரிய அளவில் செலவழிப்படக்கூடியதாக உறுதிப்படுத்த உதவுகிறது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
